# 04 — Clean CIFs: P1 Conversion & Solvent Removal (SAMOSA)

**Purpose:** Convert raw CIF files to P1 symmetry, then use SAMOSA to remove guest solvents from the framework structures.

**Requires:** CSD Python API, pymatgen, mendeleev, git. Run in CSD Python kernel.

**Inputs:**
- Raw CIF files in a user-specified input directory
- `scripts/clean_csd_mofs.py` — P1 converter script (uses CSD API + pymatgen)
- SAMOSA repository (auto-cloned from GitHub)

**Outputs:**
- `data/samosa_pipeline/01_p1_cifs/` — P1-converted CIFs
- `data/samosa_pipeline/02_samosa_output/MOFs_removed_solvent/` — cleaned CIFs (framework only)
- `data/samosa_pipeline/reports/p1_conversion_report.csv` — conversion audit
- `data/samosa_pipeline/02_samosa_output/Solvent_removal_results.csv` — SAMOSA QC report

**Pipeline:** Input CIFs → P1 conversion (expand asymmetric unit, apply symmetry ops, remove boundary duplicates) → SAMOSA (identify and remove guest molecules, counterions) → cleaned framework CIFs

## Configuration

Set the input CIF directory and runtime parameters here. Toggle `OVERWRITE_P1` and `OVERWRITE_SAMOSA_OUTPUT` to force reprocessing of already-converted files.

In [1]:
from pathlib import Path
from datetime import datetime
import importlib, os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display

# ── User configuration ───────────────────────────────────────────────────────
INPUT_CIF_DIR = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify")

PROJECT_ROOT   = Path.cwd()
SAMOSA_REPO    = PROJECT_ROOT / "external" / "SAMOSA"
P1_SCRIPT      = PROJECT_ROOT / "scripts" / "clean_csd_mofs.py"
PIPELINE_ROOT  = PROJECT_ROOT / "data" / "samosa_pipeline"
P1_OUTPUT_DIR  = PIPELINE_ROOT / "01_p1_cifs_MS"
SAMOSA_OUT_DIR = PIPELINE_ROOT / "02_samosa_output"
REPORTS_DIR    = PIPELINE_ROOT / "reports"

# Runtime controls
OVERWRITE_P1 = True
OVERWRITE_SAMOSA_OUTPUT = False
N_PROCESSES = max(1, min(8, (os.cpu_count() or 4) - 1))
REMOVABLE_DENTICITY = 1
KEEP_BOUND = False
KEEP_OXO = False
VERBOSE = True

for d in [PIPELINE_ROOT, P1_OUTPUT_DIR, SAMOSA_OUT_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.executable}")
print(f"Input CIFs: {INPUT_CIF_DIR}")
assert INPUT_CIF_DIR.exists(), f"Input directory not found: {INPUT_CIF_DIR}"
assert P1_SCRIPT.exists(), f"P1 script not found: {P1_SCRIPT}"
assert shutil.which("git"), "git not found on PATH" 

Python: c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe
Input CIFs: C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify


## 1. Prepare SAMOSA

Clones or updates the SAMOSA repository and verifies all Python dependencies are available.

In [2]:
def run_cmd(cmd, cwd=None, check=True, show=True):
    cmd = [str(x) for x in cmd]
    if show: print(f"$ {' '.join(cmd)}")
    r = subprocess.run(cmd, cwd=str(cwd) if cwd else None, capture_output=True, text=True)
    if show and r.stdout.strip(): print(r.stdout.strip())
    if show and r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}): {' '.join(cmd)}")
    return r

if SAMOSA_REPO.exists():
    run_cmd(["git", "-C", SAMOSA_REPO, "pull", "--ff-only"], check=False)
else:
    run_cmd(["git", "clone", "https://github.com/uowoolab/SAMOSA.git", SAMOSA_REPO])

assert (SAMOSA_REPO / "main.py").exists(), "SAMOSA main.py not found after clone"

# Check dependencies
for mod in ["pandas", "pymatgen", "mendeleev", "ccdc"]:
    if importlib.util.find_spec(mod) is None:
        print(f"WARNING: {mod} not available")
    else:
        print(f"OK: {mod}")

$ git -C c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML_Project_Directory\Data Collection\external\SAMOSA pull --ff-only
Already up to date.
OK: pandas
OK: pymatgen
OK: mendeleev
OK: ccdc


## 2. Discover Input CIFs

In [3]:
input_cifs = sorted(INPUT_CIF_DIR.glob("*.cif"))
print(f"Found {len(input_cifs)} CIF files")
assert input_cifs, "No CIF files in input directory"

display(pd.DataFrame({
    "file": [p.name for p in input_cifs],
    "size_kb": [round(p.stat().st_size/1024, 2) for p in input_cifs],
}).head(20))

Found 33 CIF files


,file,size_kb
0,BAHGUN02.cif,12.80
1,BUSQIQ.cif,13.67
2,ECIWUJ.cif,13.06
3,ESIPUR.cif,17.11
4,EWUCOP.cif,15.66
5,GATHAL.cif,11.10
6,GUPLEJ.cif,10.98
7,HEBZAR.cif,12.97
8,HEBZEV.cif,14.76
9,HESVOR.cif,10.05


## 3. Convert CIFs to P1 Symmetry

Runs `scripts/clean_csd_mofs.py` for each input CIF. The script:
1. Reads the asymmetric unit from the CSD entry (or CIF file directly)
2. Applies all space-group symmetry operations to generate the full unit cell
3. Removes duplicate atoms at fractional coordinate boundaries (0 vs 1)
4. Writes a P1 (triclinic, no symmetry) CIF via pymatgen

Existing P1 files are skipped unless `OVERWRITE_P1=True`.

In [4]:
if OVERWRITE_P1:
    for p in P1_OUTPUT_DIR.glob("*_P1.cif"): p.unlink()

rows = []
t0 = datetime.now()
for idx, cif in enumerate(input_cifs, 1):
    out = P1_OUTPUT_DIR / f"{cif.stem}_P1.cif"
    if out.exists() and not OVERWRITE_P1:
        rows.append({"input": cif.name, "output": out.name, "status": "skipped_existing"})
        continue

    r = run_cmd([sys.executable, P1_SCRIPT, cif.name,
                 "--read_dir", INPUT_CIF_DIR, "--write_dir", P1_OUTPUT_DIR, "-inp_is_cif"],
                check=False, show=False)
    ok = r.returncode == 0 and out.exists()
    rows.append({"input": cif.name, "output": out.name, "status": "ok" if ok else "failed",
                 "stderr": (r.stderr or "").strip()[-200:]})
    if not ok: print(f"FAILED: {cif.name}")
    if idx % 10 == 0 or idx == len(input_cifs): print(f"Progress: {idx}/{len(input_cifs)}")

report = pd.DataFrame(rows)
report.to_csv(REPORTS_DIR / "p1_conversion_report.csv", index=False)
print(f"\nDone in {datetime.now()-t0}. Status: {report['status'].value_counts().to_dict()}")

Progress: 10/33
Progress: 20/33
Progress: 30/33
Progress: 33/33

Done in 0:07:27.739023. Status: {'ok': 33}


## 4. Run SAMOSA Solvent Removal

Executes SAMOSA `main.py` in batch mode on all P1-converted CIFs. SAMOSA identifies guest molecules (solvents, counterions) and removes them, leaving only the framework structure. Outputs:
- Cleaned CIFs in `MOFs_removed_solvent/`
- QC summary in `Solvent_removal_results.csv`

In [5]:
p1_cifs = sorted(P1_OUTPUT_DIR.glob("*_P1.cif"))
print(f"P1 CIFs for SAMOSA: {len(p1_cifs)}")
assert p1_cifs, "No P1 CIFs found — run the conversion cell first"

if OVERWRITE_SAMOSA_OUTPUT and SAMOSA_OUT_DIR.exists():
    shutil.rmtree(SAMOSA_OUT_DIR); SAMOSA_OUT_DIR.mkdir(parents=True)

cmd = [sys.executable, SAMOSA_REPO / "main.py",
       "--files_path", P1_OUTPUT_DIR, "--export_path", SAMOSA_OUT_DIR,
       "--n_processes", str(N_PROCESSES), "--removable_denticity", str(REMOVABLE_DENTICITY),
       "--logging", "INFO"]
if VERBOSE: cmd.append("--verbose")
if KEEP_BOUND: cmd.append("--keep_bound")
if KEEP_OXO: cmd.append("--keep_oxo")

r = run_cmd(cmd, cwd=SAMOSA_REPO, check=False)
if r.returncode != 0:
    raise RuntimeError("SAMOSA failed — inspect output above")
print("SAMOSA completed successfully")

P1 CIFs for SAMOSA: 33
$ c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML_Project_Directory\Data Collection\external\SAMOSA\main.py --files_path c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML_Project_Directory\Data Collection\data\samosa_pipeline\01_p1_cifs_MS --export_path c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML_Project_Directory\Data Collection\data\samosa_pipeline\02_samosa_output --n_processes 7 --removable_denticity 1 --logging INFO --verbose
Identified free solvent:
[['O15'], ['O16'], ['O17'], ['O18'], ['O19'], ['O20']]
-----
Number of molecules: 6
-----
*********************
No solvent or counterions identified
*********************
Identified free solvent:
[['O17', 'H33', 'H37'], ['O18', 'H34', 'H38'], ['O19', 'H35', 'H39'], ['O20', 'H36', 'H40'], ['O21', 'H41', 'H45'], ['O22', 'H42', '

## 5. Inspect Results

Reads the SAMOSA summary CSV and lists the cleaned output CIFs with their QC flags.

In [ ]:
removed_dir = SAMOSA_OUT_DIR / "MOFs_removed_solvent"
csv_candidates = [SAMOSA_OUT_DIR / "Solvent_removal_results.csv",
                  SAMOSA_OUT_DIR / "Free_solvent_removal_results.csv"]
result_csv = next((p for p in csv_candidates if p.exists()), None)

cleaned = sorted(removed_dir.glob("*.cif")) if removed_dir.exists() else []
print(f"Cleaned CIF count: {len(cleaned)}")

if result_csv:
    stats = pd.read_csv(result_csv)
    print(f"SAMOSA summary: {len(stats)} rows")
    if "Solvent" in stats.columns:
        print(f"\nSolvent flag distribution:\n{stats['Solvent'].value_counts()}")
    display(stats)
else:
    print("No SAMOSA summary CSV found")